### Model Training
#### This notebook is to train each model and evaluate which one is the best using MLFlow
#### Firstly load data and model from notebook 2

In [4]:
import numpy as np
import joblib
import pandas as pd
from imblearn.over_sampling import SMOTE  

# Load the preprocessed data from .npy files
X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test = np.load('../data/processed/y_test.npy')
network_dataset_encoder = joblib.load('../models/label_encoder.pkl')
network_dataset_scaler = joblib.load('../models/network_dataset_scaler.pkl')

# Test the shapes of the loaded data
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

# Test y_train distribution

print('y_train distribution:')
print(pd.Series(network_dataset_encoder.inverse_transform(y_train)).value_counts())

# Test the statistics of X_train
print(f'Mean: {X_train.mean():.4f}')
print(f'Std: {X_train.std():.4f}')
print(f'Min: {X_train.min():.4f}')
print(f'Max: {X_train.max():.4f}')

X_train shape: (2016638, 51)
X_test shape: (504160, 51)
y_train shape: (2016638,)
y_test shape: (504160,)
y_train distribution:
BENIGN                        1676045
DoS Hulk                       138277
DDoS                           102411
PortScan                        72555
DoS GoldenEye                    8229
FTP-Patator                      4745
DoS slowloris                    4308
DoS Slowhttptest                 4182
SSH-Patator                      2575
Bot                              1558
Web Attack - Brute Force         1176
Web Attack - XSS                  522
Infiltration                       29
Web Attack - Sql Injection         17
Heartbleed                          9
Name: count, dtype: int64
Mean: -0.0000
Std: 1.0000
Min: -1293.9682
Max: 1164.5381


I will now look at applying SMOTE to attacks that are balanced below 1000 readings as this is the natural gap and bring them up to the next natural gap of 1558
I have used the k_neighbours of 8 as the lowest class being 9

In [13]:
# Attacks distribution before SMOTE
print('y_train distribution before SMOTE:')
print(pd.Series(network_dataset_encoder.inverse_transform(y_train)).value_counts())

# Get value counts of the encoded labels
class_distribution = pd.Series(y_train).value_counts().sort_index()


# Create sampling strategy: bring minority classes up to 1558
sampling_threshold = 1000
sampling_target = 1558
sampling_strat = {}
for label, count in class_distribution.items():
    if count < sampling_threshold:
        sampling_strat[label] = sampling_target




smote = SMOTE(random_state=42, k_neighbors=8, sampling_strategy=sampling_strat)  # Adjust k_neighbors based on the number of samples in the minority class

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)



# Attacks distribution after SMOTE
print('y_train distribution after SMOTE:')
print(pd.Series(network_dataset_encoder.inverse_transform(y_train_resampled)).value_counts())

print(f'X_train_resampled shape: {X_train_resampled.shape}')
print(f'y_train_resampled shape: {y_train_resampled.shape}')

np.save('../data/processed/X_train_resampled.npy', X_train_resampled)
np.save('../data/processed/y_train_resampled.npy', y_train_resampled)

y_train distribution before SMOTE:
BENIGN                        1676045
DoS Hulk                       138277
DDoS                           102411
PortScan                        72555
DoS GoldenEye                    8229
FTP-Patator                      4745
DoS slowloris                    4308
DoS Slowhttptest                 4182
SSH-Patator                      2575
Bot                              1558
Web Attack - Brute Force         1176
Web Attack - XSS                  522
Infiltration                       29
Web Attack - Sql Injection         17
Heartbleed                          9
Name: count, dtype: int64
y_train distribution after SMOTE:
BENIGN                        1676045
DoS Hulk                       138277
DDoS                           102411
PortScan                        72555
DoS GoldenEye                    8229
FTP-Patator                      4745
DoS slowloris                    4308
DoS Slowhttptest                 4182
SSH-Patator                    